# Lab 3 · Data-quality gate

A lightweight gate the pipeline runs **before** building silver/gold. It checks the raw meal logs and exits `PASS`/`FAIL`, so the pipeline can branch on data health.

> **Attach** the `lh_resident360` Lakehouse first.

In [ ]:
from pyspark.sql import functions as F
import json, notebookutils

df = spark.table("bronze.h365_meal_logs")
total = df.count()
bad_dates = df.filter(F.to_date("log_date").isNull()).count()
null_cal = df.filter(F.col("calories").isNull() | (F.trim("calories") == "")).count()
bad_pct = round(100.0 * (bad_dates + null_cal) / max(total, 1), 2)
status = "PASS" if bad_pct < 5.0 else "FAIL"
result = {"status": status, "total": total, "bad_dates": bad_dates,
          "null_calories": null_cal, "bad_pct": bad_pct}
print(result)
notebookutils.notebook.exit(json.dumps(result))